# Learn AutoGluon in 60 Minutes

## Learning goals
By the end of this guide, you should be able to:
1. Explain what AutoGluon automates and what you still control.
2. Create a `TabularPredictor` and fit it on labeled tabular data.
3. Predict unseen prices, compare models, and evaluate errors.
4. Calculate permutation feature importance and interpret its limitations.
5. Save/load a trained predictor.


## 1. What is AutoGluon?
AutoGluon is an AutoML library that can fit and compare multiple algorithms, automate their preprocessing and tuning, and combine models into an ensemble. Unlike a hand-written scikit-learn pipeline, AutoGluon can infer how to handle many categorical and text features, reducing amount of preprocessing necessary. 

## 2. API quick reference
| API | Purpose | Important parameter(s) |
|---|---|---|
| `TabularPredictor(...)` | Define prediction problem | `label`, `problem_type`, `eval_metric`, `path` |
| `.fit(train_data, ...)` | Train candidate models and ensemble | `presets`, `time_limit` |
| `.predict(features)` | Predict target for new rows | Feature columns |
| `.evaluate(test_data)` | Evaluate labeled test observations | Data must include `label` |
| `.leaderboard(test_data)` | Compare trained model performance | Optional test data |
| `.feature_importance(test_data)` | Permutation-based importance | Labeled evaluation dataset |
| `TabularPredictor.load(path)` | Reopen saved model | Predictor directory |

**A note on metrics:** In AutoGluon, MAE-based leaderboard values may be represented in a higher-is-better form (negative MAE). A *less negative* score is better; use standard MAE calculated with scikit-learn when reporting dollar errors.

## 3. Minimal working demonstration
This example is runnable without downloading the Airbnb dataset, as it is designed for learning the API quickly.

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
from autogluon.tabular import TabularPredictor

rng = np.random.default_rng(42)
n = 240
rooms = rng.choice(["Entire home/apt", "Private room"], size=n)
boroughs = rng.choice(["Manhattan", "Brooklyn", "Queens"], size=n)
price = (55 + 100 * (rooms == "Entire home/apt") + 55 * (boroughs == "Manhattan")
         + rng.normal(0, 18, size=n))
demo = pd.DataFrame({"room_type": rooms, "borough": boroughs, "minimum_nights": rng.integers(1, 15, size=n), "price": price})
train_demo = demo.sample(frac=0.8, random_state=42)
test_demo = demo.drop(train_demo.index)
print(train_demo.shape, test_demo.shape)

(192, 4) (48, 4)


### `TabularPredictor` and `.fit()`
`label="price"` selects the target. `problem_type="regression"` predicts a numeric quantity. `eval_metric="mean_absolute_error"` prioritizes dollar-scale errors. `path` specifies where the predictor saves its artifacts. `time_limit` keeps this teaching example relatively short. It can also lead to different trained model sets.

In [7]:
demo_path = "AutogluonModels/tutorial_demo"
demo_predictor = TabularPredictor(
    label="price",
    problem_type="regression",
    eval_metric="mean_absolute_error",
    path=demo_path,
).fit(train_data=train_demo, time_limit=90)

Verbosity: 2 (Standard Logging)
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.11.18
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Tue Oct  6 19:13:07 UTC 2026
CPU Count:          8
Pytorch Version:    Can't import torch
CUDA Version:       Can't get cuda version from torch
Memory Avail:       2.61 GB / 3.78 GB (69.0%)
Disk Space Avail:   3.88 GB / 236.34 GB (1.6%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme' 

### `.predict()`, `.evaluate()`, and `.leaderboard()`
`.predict()` uses **only the features** to produce target predictions on the test data. Does not require explicitly dropping the target column from the test data; AutoGluon will ignore it if present. `.evaluate()` computes how well the model was able to make predictions across several metrics. `.leaderboard()` provides a comparison table of the performances of the different models that AutoGluon explored.

`.evaluate()` requires labeled test data. Do not confuse the leaderboard's internal validation performance with unseen test performance.

In [8]:
import sys
from importlib.metadata import version

print("Python:", sys.version)
print("AutoGluon:", version("autogluon.tabular"))
print("Kernel works:", 2 + 2)

Python: 3.11.18 (main, Oct 28 2026, 02:02:50) [GCC 14.2.0]
AutoGluon: 1.6.3
Kernel works: 4


In [3]:
demo_preds = demo_predictor.predict(test_demo)
print("First predictions:", demo_preds.head())
print("Test evaluation:", demo_predictor.evaluate(test_demo))
display(demo_predictor.leaderboard(test_demo, silent=True).head())

First predictions: 1      39.426701
14     54.892670
20     37.607624
21    189.190323
37     37.607624
Name: price, dtype: float32
Test evaluation: {'mean_absolute_error': -17.649870475919453, 'root_mean_squared_error': np.float64(-21.115859999383282), 'mean_squared_error': -445.8795435135549, 'r2': 0.8817162050247807, 'pearsonr': 0.9464341346899123, 'median_absolute_error': -17.380481773113566}


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,RandomForestMSE,-17.649870,-16.629801,mean_absolute_error,0.397751,0.213346,2.326733,0.397751,0.213346,2.326733,1,True,1
1,WeightedEnsemble_L2,-17.649870,-16.629801,mean_absolute_error,0.411418,0.215972,2.390198,0.013667,0.002626,0.063465,2,True,3
2,ExtraTreesMSE,-17.760245,-16.709711,mean_absolute_error,0.386028,0.143394,1.170641,0.386028,0.143394,1.170641,1,True,2


### `.feature_importance()`
Feature importance measures how much predictive performance worsens when a feature’s values are shuffled. It measures predictive dependence, not causation. Larger positive values indicate greater importance, while values near zero mean shuffling had little effect. Negative values mean performance improved after shuffling, which may suggest noise or a potentially harmful feature. Correlated features like square footage and number of bedrooms, can substitute for each other, making their individual importance seem lower.

In [4]:
demo_importance = demo_predictor.feature_importance(test_demo, num_shuffle_sets=3, silent=True)
display(demo_importance)

,importance,stddev,p_value,n,p99_high,p99_low
room_type,40.278302,5.933909,0.003579,3,74.280256,6.276348
borough,17.892167,4.968014,0.012375,3,46.359436,-10.575102
minimum_nights,-0.354103,0.813935,0.735120,3,4.309834,-5.018039


### Model persistence
The predictor saves its artifacts to previously specified path: `demo_path`. You can reload the predictor as long as the files still exist. For a new experiment, it is suggested to create a new path, as re-running `.fit()` at an existing path can cause conflicts with old artifacts.

In [5]:
reloaded = TabularPredictor.load(demo_path)
assert len(reloaded.predict(test_demo.drop(columns="price"))) == len(test_demo)
print("Reload successful")

Reload successful


## Checkpoint questions
1. What does `.predict()` indicate? What does a greater value signify?
2. What is the difference between `.evaluate()` and `.leaderboard()`?
3. What changes when you decrease `time_limit`?
4. Train a new predictor using `eval_metric="root_mean_squared_error"`. Keep the same training data and time limit, and save it to a new path `AutogluonModels/rmseCheckpoint`. Which three models have the best validation scores? Compare their ranking with the original MAE-based leaderboard and discuss any differences. **Feel free to use the cells below**

**Answer Here:**


**Next:** Open `autogluon.example.ipynb` to apply these methods to real NYC Airbnb data.